# Dental Bill Detective v2 — Setup Notebook

**4 parts:**
1. Create Contextual AI datastore + agent
2. Build + launch Docker services
3. Test a sample bill query
4. Send a Telegram test message

In [ ]:
import os, sys
sys.path.insert(0, '../openclaw')
from dotenv import load_dotenv
load_dotenv('../.env')
print('Env loaded. Keys present:', [k for k in ['CONTEXTUAL_API_KEY','APIFY_API_TOKEN','ANTHROPIC_API_KEY','TELEGRAM_BOT_TOKEN','CIVIC_API_KEY','REDIS_URL'] if os.environ.get(k)])

## Part 1: Create Contextual AI Datastore + Agent

Run this once. Copy the IDs into your `.env` file.

In [ ]:
from contextual import ContextualAI

client = ContextualAI(api_key=os.environ['CONTEXTUAL_API_KEY'])

# Create datastore
ds = client.datastores.create(name='Dental Pricing Benchmarks v2')
print(f'Datastore created: {ds.id}')
print(f'>>> Add to .env: DATASTORE_ID={ds.id}')

In [ ]:
DATASTORE_ID = ds.id  # or paste manually: DATASTORE_ID = 'your_id_here'

agent = client.agents.create(
    name='Dental Bill Auditor v2',
    datastore_ids=[DATASTORE_ID],
    system_prompt=(
        'You are an expert dental billing auditor with deep knowledge of CDT procedure codes, '
        'FairHealth consumer cost benchmarks, and CMS Medicare fee schedules. '
        'When given a list of billed CDT codes and amounts, identify: '
        '(1) overcharges above the 80th percentile benchmark, '
        '(2) duplicate billing of the same code, '
        '(3) unbundled codes that should be billed as one, '
        '(4) upcoded procedures where a simpler code applies. '
        'Always respond in the exact JSON format specified in the query.'
    ),
)
print(f'Agent created: {agent.id}')
print(f'>>> Add to .env: AGENT_ID={agent.id}')

## Part 2: Build + Launch Docker Services

In [ ]:
import subprocess

print('Building Docker images...')
result = subprocess.run(
    ['docker', 'compose', 'build', '--no-cache'],
    cwd='..', capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else '')
if result.returncode != 0:
    print('BUILD ERROR:', result.stderr[-1000:])
else:
    print('Build successful!')

In [ ]:
print('Starting services...')
result = subprocess.run(
    ['docker', 'compose', 'up', '-d'],
    cwd='..', capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)
else:
    print('Services started! Check logs with: docker compose logs -f')

## Part 3: Test a Sample Bill Query

This tests the full pipeline without Telegram: OCR → CDT extract → RAG agent → report.

In [ ]:
import json
import requests

CONTEXTUAL_API_KEY = os.environ['CONTEXTUAL_API_KEY']
AGENT_ID = os.environ.get('AGENT_ID', '')

if not AGENT_ID:
    print('ERROR: AGENT_ID not set in .env. Run Part 1 first.')
else:
    # Test query via REST API
    sample_prompt = """
    Analyze this dental bill:
    - D2740: $1,800.00 (Crown - porcelain)
    - D0120: $150.00 (Periodic oral evaluation)
    - D1110: $250.00 (Adult cleaning)
    - D2740: $1,800.00 (Crown - porcelain) [DUPLICATE]

    For each CDT code, check against FairHealth 80th percentile benchmarks.
    Respond in JSON format with summary, line_items, and phone_script.
    """

    resp = requests.post(
        f'https://api.contextual.ai/v1/agents/{AGENT_ID}/query',
        headers={
            'Authorization': f'Bearer {CONTEXTUAL_API_KEY}',
            'Content-Type': 'application/json',
        },
        json={'messages': [{'role': 'user', 'content': sample_prompt}]},
        timeout=60,
    )
    print(f'Status: {resp.status_code}')
    if resp.status_code == 200:
        answer = resp.json()['message']['content']
        print('Agent response:')
        print(answer[:2000])
    else:
        print('Error:', resp.text[:500])

In [ ]:
# Also test via Python SDK
from contextual import ContextualAI

c = ContextualAI(api_key=os.environ['CONTEXTUAL_API_KEY'])

# IMPORTANT: use .create() not .query()
sdk_resp = c.agents.query.create(
    agent_id=AGENT_ID,
    messages=[{'role': 'user', 'content': 'What CDT codes are most commonly overbilled?'}]
)
print('SDK response content:')
print(sdk_resp.message.content[:1000])

## Part 4: Send Telegram Test Message

In [ ]:
TELEGRAM_BOT_TOKEN = os.environ.get('TELEGRAM_BOT_TOKEN', '')
TELEGRAM_CHAT_ID = os.environ.get('TELEGRAM_CHAT_ID', '')

if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
    print('Set TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID in .env first')
else:
    resp = requests.post(
        f'https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage',
        json={
            'chat_id': TELEGRAM_CHAT_ID,
            'text': (
                'Dental Bill Detective v2 is live!\n\n'
                'Send me a photo or PDF of your dental bill and I will '
                'check it for overcharges against FairHealth + CMS benchmarks.\n\n'
                'Powered by Contextual AI + Apify + Civic + Redis + Claude'
            ),
        },
    )
    print(f'Telegram send status: {resp.status_code}')
    print(resp.json())